In [ ]:
"""
Combine processed A/B recordings into a single multichannel 'ref' WAV and
copy diarisation + transitional tiers into a new TextGrid in Data_FINAL.

  * No resampling, no normalization, no gain changes.
  * If A/B lengths differ slightly, the shorter is zero-padded.
  * Tier intervals are clipped to match WAV duration.
"""

import os
import re
import json
import traceback
from pathlib import Path
from typing import Optional, List, Tuple

import numpy as np
import soundfile as sf
from tqdm import tqdm
from praatio import tgio

REC_DIR   = Path("/Users/moanason/Downloads/Data_REC")
REF_DIR   = Path("/Users/moanason/Downloads/Data_REF")
FINAL_DIR = Path("/Users/moanason/Downloads/Data_TEMP") # changed from Data_FINAL to Data_TEMP for testing/updating

AB_RX = re.compile(
    r"^(?P<prefix>p\d+)_s(?P<sess>\d{2})_(?P<ab>[AB])_NC(?P<nc>\d+)_processed\.wav$",
    re.IGNORECASE,
)

REF_TG_BASENAME = "rec_s{sess}_NC{nc}.TextGrid"
REF_TG_BASENAME_PROCESSED = "rec_s{sess}_NC{nc}_processed.TextGrid"  # fallback

FINAL_WAV_NAME = "ref_s{sess}_NC{nc}_processed.wav"
FINAL_TG_NAME  = "ref_s{sess}_NC{nc}_processed.TextGrid"

TIER_A = "Diarisation_A"
TIER_B = "Diarisation_B"
TIER_TRANS = "TransCondition"
TIER_TEVENTS = "TransEvents"

MAX_LEN_DIFF_SAMPLES_WARN = 2400   # ~50 ms at 48kHz; warn if exceeded (still pad)
ABORT_IF_SR_MISMATCH = True        # error/skip if A and B have different sample rates

# ------------------------------------------------


def read_wav(path: Path) -> Tuple[np.ndarray, int]:
    x, sr = sf.read(str(path), always_2d=True)
    return x.astype(np.float32), int(sr)


def write_wav(path: Path, x: np.ndarray, sr: int) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), x.astype(np.float32), sr)


def list_ab_pairs(rec_dir: Path):
    sessions = sorted([p for p in rec_dir.iterdir() if p.is_dir() and re.fullmatch(r"s\d{2}", p.name)])
    for sess_dir in sessions:
        wavs = sorted(sess_dir.glob("*.wav"))
        # index by (sess,nc)->A/B paths
        by_key = {}
        for w in wavs:
            m = AB_RX.match(w.name)
            if not m:
                continue
            sess = m.group("sess")
            nc   = m.group("nc")
            ab   = m.group("ab").upper()
            key = (sess, nc)
            d = by_key.setdefault(key, {"A": None, "B": None})
            d[ab] = w
        # yield complete pairs
        for (sess, nc), d in sorted(by_key.items()):
            if d["A"] is not None and d["B"] is not None:
                yield sess, nc, d["A"], d["B"]


def pad_to_same_length(a: np.ndarray, b: np.ndarray) -> Tuple[np.ndarray, np.ndarray, int]:
    # pad
    na, nb = len(a), len(b)
    n = max(na, nb)
    if na < n:
        pad = np.zeros((n - na, a.shape[1]), dtype=a.dtype)
        a = np.vstack([a, pad])
    if nb < n:
        pad = np.zeros((n - nb, b.shape[1]), dtype=b.dtype)
        b = np.vstack([b, pad])
    return a, b, n


def combine_channels(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    return np.concatenate([a, b], axis=1)


# for tg handling
def open_textgrid_safe(path: Path) -> Optional[tgio.Textgrid]:
    if not path.exists():
        return None
    try:
        return tgio.openTextgrid(str(path))
    except Exception:
        return None


def get_tier_if_exists(tg: tgio.Textgrid, name: str):
    try:
        return tg.tierDict[name] if name in tg.tierDict else None
    except Exception:
        try:
            return tg.getTier(name)
        except Exception:
            return None


def tier_entries(tier) -> List[Tuple[float, float, str]]:
    if tier is None:
        return []
    # praatio IntervalTier stores entries in .entryList
    entries = getattr(tier, "entryList", None)
    if entries is None:
        # fallback if older attr name
        entries = getattr(tier, "entries", None)
    if entries is None:
        return []
    out = []
    for e in entries:
        if len(e) >= 3:
            out.append((float(e[0]), float(e[1]), str(e[2])))
    return out


def clip_entries_to_duration(entries: List[Tuple[float, float, str]], total_dur: float) -> List[Tuple[float, float, str]]:
    out = []
    for s, e, lab in entries:
        s2 = max(0.0, float(s))
        e2 = min(float(total_dur), float(e))
        if e2 > s2:
            out.append((s2, e2, lab))
    return out


def build_textgrid_with_three_tiers(
    total_dur: float,
    diarA: List[Tuple[float, float, str]],
    diarB: List[Tuple[float, float, str]],
    trans: List[Tuple[float, float, str]],
) -> tgio.Textgrid:
    tg = tgio.Textgrid()
    tier_a = tgio.IntervalTier(TIER_A, diarA, 0.0, float(total_dur))
    tier_b = tgio.IntervalTier(TIER_B, diarB, 0.0, float(total_dur))
    tier_t = tgio.IntervalTier(TIER_TRANS, trans, 0.0, float(total_dur))
    # tier_te = tgio.IntervalTier(TIER_TEVENTS, trans, 0.0, float(total_dur))
    tg.addTier(tier_a)
    tg.addTier(tier_b)
    tg.addTier(tier_t)
    # tg.addTier(tier_te)
    return tg


def read_ref_tiers(sess: str, nc: str) -> Tuple[List[Tuple[float, float, str]],
                                                List[Tuple[float, float, str]],
                                                List[Tuple[float, float, str]]]:
    cand = [
        REF_DIR / REF_TG_BASENAME.format(sess=sess, nc=nc),
        REF_DIR / REF_TG_BASENAME_PROCESSED.format(sess=sess, nc=nc),
    ]
    tg = None
    for p in cand:
        tg = open_textgrid_safe(p)
        if tg is not None:
            break
    if tg is None:
        return [], [], []

    # prefer exact tier names
    tierA = get_tier_if_exists(tg, TIER_A)
    tierB = get_tier_if_exists(tg, TIER_B)
    tierT = get_tier_if_exists(tg, TIER_TRANS)

    if tierA is None or tierB is None:
        dia = get_tier_if_exists(tg, "Diarisation")
        if dia is not None:
            entries = tier_entries(dia)
            a_list, b_list = [], []
            for s, e, lab in entries:
                # heuristics:labels containing '_A' vs '_B' (e.g., s03_A, s03_B)
                u = lab.upper()
                if u.endswith("_A"):
                    a_list.append((s, e, lab))
                elif u.endswith("_B"):
                    b_list.append((s, e, lab))
            if tierA is None and a_list:
                tierA = tgio.IntervalTier(TIER_A, a_list, 0.0, dia.maxTimestamp)
            if tierB is None and b_list:
                tierB = tgio.IntervalTier(TIER_B, b_list, 0.0, dia.maxTimestamp)

    diarA = tier_entries(tierA)
    diarB = tier_entries(tierB)
    trans = tier_entries(tierT)
    return diarA, diarB, trans


def read_separated_tiers(a_tg_path: Path, b_tg_path: Path) -> Tuple[List[Tuple[float, float, str]],
                                                                    List[Tuple[float, float, str]]]:
    outA, outB = [], []

    def pull_one(p: Path, want_ab: str) -> List[Tuple[float, float, str]]:
        tg = open_textgrid_safe(p)
        if tg is None:
            return []
        # try 'Diarisation_A' / 'Diarisation_B'
        tier = get_tier_if_exists(tg, f"Diarisation_{want_ab}")
        if tier is not None:
            return tier_entries(tier)
        # fallback single 'Diarisation'
        tier = get_tier_if_exists(tg, "Diarisation")
        if tier is not None:
            return tier_entries(tier)
        return []

    outA = pull_one(a_tg_path, "A")
    outB = pull_one(b_tg_path, "B")
    return outA, outB


def make_final_for_pair(sess: str, nc: str, a_wav: Path, b_wav: Path, logs: list):
    xA, srA = read_wav(a_wav)
    xB, srB = read_wav(b_wav)

    if ABORT_IF_SR_MISMATCH and srA != srB:
        logs.append(f"[SKIP s{sess} NC{nc}] SR mismatch A={srA}, B={srB}")
        return False

    # length reconcile (pad shorter)
    if len(xA) != len(xB):
        if abs(len(xA) - len(xB)) > MAX_LEN_DIFF_SAMPLES_WARN:
            logs.append(f"[WARN s{sess} NC{nc}] A/B length diff {abs(len(xA)-len(xB))} samples; zero-padding shorter.")
        xA, xB, n = pad_to_same_length(xA, xB)

    # combine channels: A first, then B
    y = combine_channels(xA, xB)

    FINAL_DIR.mkdir(parents=True, exist_ok=True)
    out_wav = FINAL_DIR / FINAL_WAV_NAME.format(sess=sess, nc=nc)
    write_wav(out_wav, y, srA)
    total_dur = len(y) / float(srA)

    diarA, diarB, trans = read_ref_tiers(sess, nc)

    if not diarA or not diarB or not trans:
        a_tg = a_wav.with_suffix(".TextGrid")
        b_tg = b_wav.with_suffix(".TextGrid")
        if (not diarA) or (not diarB):
            dA_sep, dB_sep = read_separated_tiers(a_tg, b_tg)
            if not diarA:
                diarA = dA_sep
            if not diarB:
                diarB = dB_sep
        if not trans:
            trans = []

    # clip to duration
    diarA = clip_entries_to_duration(diarA, total_dur)
    diarB = clip_entries_to_duration(diarB, total_dur)
    trans = clip_entries_to_duration(trans, total_dur)

    # build
    tg = build_textgrid_with_three_tiers(total_dur, diarA, diarB, trans)

    out_tg = FINAL_DIR / FINAL_TG_NAME.format(sess=sess, nc=nc)
    tg.save(str(out_tg), minimumIntervalLength=0.0, outputFormat="textgrid")

    logs.append(f"[OK  s{sess} NC{nc}] -> {out_wav.name}, {out_tg.name}")
    return True


def main():
    pairs = list(list_ab_pairs(REC_DIR))
    print(f"Found {len(pairs)} A/B processed pairs in {REC_DIR}")

    ok, fail = 0, 0
    logs = []
    for sess, nc, a_wav, b_wav in tqdm(pairs, desc="Combine + Copy tiers"):
        try:
            done = make_final_for_pair(sess, nc, a_wav, b_wav, logs)
            if done:
                ok += 1
            else:
                fail += 1
        except Exception as e:
            fail += 1
            logs.append(f"[ERR s{sess} NC{nc}] {repr(e)}\n{traceback.format_exc().splitlines()[-1]}")

    print(f"\nDone. Success={ok}, Failed/Skipped={fail}")
    if logs:
        print("---- Summary ----")
        for line in logs:
            print(line)

    # Optional: write a small JSON log
    log_path = FINAL_DIR / "final_build_log.json"
    with open(log_path, "w") as f:
        json.dump({"success": ok, "failed_or_skipped": fail, "messages": logs}, f, indent=2)
    print(f"Log -> {log_path}")


if __name__ == "__main__":
    main()


Found 6 A/B processed pairs in /Users/moanason/Downloads/Data_REC


Combine + Copy tiers: 100%|██████████| 6/6 [00:03<00:00,  1.53it/s]


Done. Success=6, Failed/Skipped=0
---- Summary ----
[OK  s05 NC1] -> ref_s05_NC1_processed.wav, ref_s05_NC1_processed.TextGrid
[OK  s05 NC2] -> ref_s05_NC2_processed.wav, ref_s05_NC2_processed.TextGrid
[OK  s06 NC1] -> ref_s06_NC1_processed.wav, ref_s06_NC1_processed.TextGrid
[OK  s06 NC2] -> ref_s06_NC2_processed.wav, ref_s06_NC2_processed.TextGrid
[OK  s07 NC1] -> ref_s07_NC1_processed.wav, ref_s07_NC1_processed.TextGrid
[OK  s07 NC2] -> ref_s07_NC2_processed.wav, ref_s07_NC2_processed.TextGrid
Log -> /Users/moanason/Downloads/Data_TEMP/final_build_log.json
